# Development project 
## Cleaning Demographics, Puberty, CBCL, FES data 
### - Creating relevant columns, e.g. pds score, selecting columns of interest for demographics and CBCL, creating total score for FES without missing
### - These cleaned puberty/CBCL/FES CSV files (output) are to feed to p3_cleaning_final_sample.py (where the matching subjects is done, puberty missing values are excluded (but not missing values for supplementary analyses like parental education, CBCL and FES)

## Load Packages

In [1]:
# General
import os
import sys
import numpy as np
import pandas as pd
import csv
import math
from math import isnan
import statistics
import pickle
from collections import Counter

# Computing / Analyses
import scipy.io  # loadmat
import fnmatch # for comparing patterns of syntax
from scipy import stats
import sklearn 
from brainstat.stats.terms import FixedEffect
from brainstat.stats.SLM import SLM
from statsmodels.stats.multitest import fdrcorrection # does not yield exactly the same FDR correction as R but the same up to 14th decimal place so good enough
import statsmodels.regression.mixed_linear_model as sm
from sklearn.preprocessing import scale

# Visualisation
import matplotlib.pyplot as plt 
import seaborn as sns
import vtk
from IPython.display import display
import matplotlib.collections as clt
#import ptitprince as pt  # commented out because clashing dependencies with seaborn

# Neuroimaging
import nibabel as nib
import nilearn
from brainstat.datasets import fetch_parcellation
from enigmatoolbox.permutation_testing import spin_test, shuf_test

# Gradients
import brainspace
from brainspace.datasets import load_parcellation, load_conte69
from brainspace.plotting import plot_hemispheres
from brainspace.gradient import GradientMaps
from brainspace.utils.parcellation import map_to_labels

## Define directories

In [2]:
codedir = os.path.abspath('')  # obtain current direction from which script is runnning

datadir = '/data/pt_02667/data/ABCD/'
datadir_local = '/data/p_02667/development/data/'

resdir = '/data/p_02667/development/results/'
resdir_fig = '/data/p_02667/development/results/figures/'

# Demographic data

https://nda.nih.gov/data_structure.html?short_name=pdem02 

In [3]:
## Load dataframes 
# Demographics (parental report)
abcd_p_demo = pd.read_csv(datadir+'abcd-data-release-5.1/core/abcd-general/abcd_p_demo.csv')

# longitudinal tracking
abcd_y_lt = pd.read_csv(datadir+'abcd-data-release-5.1/core/abcd-general/abcd_y_lt.csv')

# BMI calculated from abcd_y_anthro in p3_sensitivity_computing_BMIz.R
abcd_bmi = pd.read_csv(datadir_local+'abcd_bmi.csv')

/tmp/ipykernel_4790/944744546.py:6: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  abcd_y_lt = pd.read_csv(datadir+'abcd-data-release-5.1/core/abcd-general/abcd_y_lt.csv')


In [4]:
## keeping full sample

# note: here length is larger than 4938 (3x1646) because there are also the non-imaging timepoints (1y & 3y followup data)
# here we don't have full 5x1646 data (maybe missing data from sessions 1y & 3y in some subjects) but note the NaNs: demographics probably only filled at baseline anyway

## take one entry per subject because we only need the information once
abcd_p_demo_baseline =  abcd_p_demo[abcd_p_demo['eventname'] == 'baseline_year_1_arm_1']

# taking only a subset of df with relevant columns(*** WILL LATER ADD OTHERS LIKE RACE AND SES - should also add site (AND GENDER CHILD REPORT FROM OTHER CSV FILE) IF THEY CAN BE SIMPLY ADDED HERE, OR BELOW IF NEED TO CLEAN ***)
# not taking the "baseline" column as it's in the name of the dataframe (and information that won't change (ie age at baseline, sex)
abcd_demo_clean_baseline = abcd_p_demo_baseline[['src_subject_id', 'demo_brthdat_v2', 'demo_prnt_ed_v2', 'demo_prtnr_ed_v2']]


### Cleaning variables

# renaming some variables
abcd_demo_clean_baseline = abcd_demo_clean_baseline.rename(columns={"demo_brthdat_v2": "age_baseline_yrs", "demo_prnt_ed_v2": "edu_parent_self", "demo_prtnr_ed_v2": "edu_parent_partner"})


## SEX: coding female and male
# demo_sex_v2: 'What sex was the child assigned at birth, on the original birth certificate?' -> 1 = Male; 2 = Female; 3 = Intersex-Male; 4 = Intersex-Female; 999 = Don't know; 777 = Refuse to answer 

sex_letter = []

for sex_num in abcd_p_demo_baseline.demo_sex_v2:
    
    if sex_num == 1:
        sex_letter.append('M')
        
    elif sex_num == 2:
        sex_letter.append('F')
        
    elif sex_num == 3:
        sex_letter.append('Intersex-M')
        
    elif sex_num == 4:
        sex_letter.append('Intersex-F')
        
    else:  # would represent "don't know" and "refuse to answer"
        sex_letter.append('n/a') 

abcd_demo_clean_baseline['sex'] = sex_letter


## Making a combined parental education column (highest of the two)

# Recode 777 / 999 to NaN
abcd_demo_clean_baseline[["edu_parent_self", "edu_parent_partner"]] = (abcd_demo_clean_baseline[["edu_parent_self", "edu_parent_partner"]].replace([777.0, 999.0], np.nan))

# Create edu_parent as the max across the two columns (ignores NaNs by default)
abcd_demo_clean_baseline["edu_parent"] = (abcd_demo_clean_baseline[["edu_parent_self", "edu_parent_partner"]].max(axis=1))

## reset index
abcd_demo_clean_baseline.reset_index(drop=True)

,src_subject_id,age_baseline_yrs,edu_parent_self,edu_parent_partner,sex,edu_parent
0,NDAR_INV003RTV85,10.0,13.0,13.0,F,13.0
1,NDAR_INV005V6D2C,10.0,6.0,NaN,F,6.0
2,NDAR_INV007W6H7B,10.0,19.0,18.0,M,19.0
3,NDAR_INV00BD7VDC,9.0,20.0,20.0,M,20.0
4,NDAR_INV00CY2MDM,10.0,15.0,NaN,M,15.0
...,...,...,...,...,...,...
11863,NDAR_INVZZNX6W2P,10.0,18.0,18.0,M,18.0
11864,NDAR_INVZZPKBDAC,9.0,19.0,18.0,F,19.0
11865,NDAR_INVZZZ2ALR6,10.0,21.0,21.0,F,21.0
11866,NDAR_INVZZZNB0XC,9.0,19.0,13.0,F,19.0


In [5]:
### BMI

abcd_bmi

,src_subject_id,sex,eventname,anthroheightcalc,anthroweightcalc,weight_kg,height_m,BMI,BMIz,BMIz_mod
0,NDAR_INV003RTV85,F,baseline_year_1_arm_1,56.5,93.00,42.184090,1.43510,20.482566,0.961662,0.649804
1,NDAR_INV005V6D2C,F,baseline_year_1_arm_1,56.5,100.00,45.359237,1.43510,22.024264,1.462009,1.164654
2,NDAR_INV007W6H7B,M,baseline_year_1_arm_1,56.5,82.80,37.557448,1.43510,18.236091,0.550824,0.312880
3,NDAR_INV00BD7VDC,M,baseline_year_1_arm_1,57.5,76.80,34.835894,1.46050,16.331416,0.007539,0.003779
4,NDAR_INV00CY2MDM,M,baseline_year_1_arm_1,56.5,91.50,41.503702,1.43510,20.152202,1.060263,0.702610
...,...,...,...,...,...,...,...,...,...,...
25280,NDAR_INVZZZNB0XC,F,baseline_year_1_arm_1,49.0,63.00,28.576319,1.24460,18.447890,0.836305,0.545919
25281,NDAR_INVZZZNB0XC,F,4_year_follow_up_y_arm_1,55.0,92.55,41.979974,1.39700,21.510443,0.788166,0.501856
25282,NDAR_INVZZZP87KR,F,baseline_year_1_arm_1,59.5,123.00,55.791862,1.51130,24.426964,1.721165,1.594277
25283,NDAR_INVZZZP87KR,F,2_year_follow_up_y_arm_1,63.5,164.00,74.389149,1.61290,28.595303,1.882753,1.937075


In [6]:
print(f"BMIz min: {min(abcd_bmi.BMIz)}")
print(f"BMIz max: {max(abcd_bmi.BMIz)}")
print(f"modified BMIz min: {min(abcd_bmi.BMIz_mod)}")
print(f"modified BMIz max: {max(abcd_bmi.BMIz_mod)}")

BMIz min: -1086.26513683548
BMIz max: 8.21
modified BMIz min: -10.4496748141325
modified BMIz max: 828.277721783276


In [7]:
# Use BMIz_modified to identify Biologically Implausible BMI Values (BIV): BIMz_mod < -5 and BIMz_mod > 10

biv_mask = (abcd_bmi.BMIz_mod < -5) | (abcd_bmi.BMIz_mod > 10)
biv = abcd_bmi.BMIz_mod[biv_mask]

In [8]:
abcd_bmi[biv_mask]

,src_subject_id,sex,eventname,anthroheightcalc,anthroweightcalc,weight_kg,height_m,BMI,BMIz,BMIz_mod
622,NDAR_INV0P34UPZ9,M,baseline_year_1_arm_1,4.000000,68.666667,31.146676,0.101600,3017.340279,8.210000,828.277722
1494,NDAR_INV1PHTC329,M,baseline_year_1_arm_1,70.500000,54.966667,24.932461,1.790700,7.775342,-21.492074,-6.303743
2018,NDAR_INV2BPVXTH3,F,2_year_follow_up_y_arm_1,76.000000,58.000000,26.308357,1.930400,7.059909,-18.750755,-5.866736
2494,NDAR_INV3062KMFL,F,4_year_follow_up_y_arm_1,61.000000,6201.400000,2812.907723,1.549400,1171.732247,8.210000,198.177732
2717,NDAR_INV39Z1U19N,F,2_year_follow_up_y_arm_1,5.750000,140.000000,63.502932,0.146050,2977.081018,8.210000,557.911412
3449,NDAR_INV464X4LJP,F,2_year_follow_up_y_arm_1,56.600000,355.333333,161.176489,1.437640,77.983262,8.210000,11.339094
3542,NDAR_INV4BAP6ERL,M,baseline_year_1_arm_1,60.000000,11.000000,4.989516,1.524000,2.148268,-909.126223,-10.449675
4648,NDAR_INV5RAR4VBD,F,4_year_follow_up_y_arm_1,5.050000,103.333333,46.871212,0.128270,2848.760836,8.210000,515.225978
7997,NDAR_INVA0BVG0DF,M,baseline_year_1_arm_1,56.600000,24.000000,10.886217,1.437640,5.267162,-64.294295,-7.918818
8225,NDAR_INVA9CA7RPB,M,baseline_year_1_arm_1,59.250000,37.500000,17.009714,1.504950,7.510224,-23.930369,-6.490297


In [9]:
len(abcd_bmi[biv_mask])

31

In [10]:
# Code to NaN BMIz values that have BIV for BMIz_mod 
abcd_bmi.loc[biv_mask, "BMIz"] = np.nan

print(f"BMIz min: {min(abcd_bmi.BMIz)}")
print(f"BMIz max: {max(abcd_bmi.BMIz)}")

BMIz min: -9.9353760166949
BMIz max: 7.33228371419966


In [11]:
### Family and site information

# take a subset of the dataframe - i.e. only the events we need: baseline, 2y and 4y follow up
abcd_y_lt_full = abcd_y_lt[abcd_y_lt['eventname'].isin(['baseline_year_1_arm_1', '2_year_follow_up_y_arm_1', '4_year_follow_up_y_arm_1'])]

# taking only a subset of df with relevant columns
abcd_y_lt_clean = abcd_y_lt_full[['src_subject_id', 'eventname', 'site_id_l', 'rel_family_id', 'interview_age']]


# Given that family ID info is only available at baseline, we need to populate family id for other timepoints as well

# Step 1: Extract rel_family_id from baseline only
baseline_rel_fam = abcd_y_lt_clean[abcd_y_lt_clean['eventname'] == 'baseline_year_1_arm_1'][['src_subject_id', 'rel_family_id']]

# Step 2: Merge rel_family_id from baseline into the full dataframe using src_subject_id as key and rename rel_family_id to re_family_id
abcd_y_lt_clean = (
    abcd_y_lt_clean
    .drop(columns='rel_family_id')
    .merge(baseline_rel_fam, on='src_subject_id', how='left')
    .rename(columns={'rel_family_id': 'family_id'})
)


# merge BMI column into abcd_y_lt_clean
abcd_y_lt_clean = abcd_y_lt_clean.merge(
    abcd_bmi[["src_subject_id", "eventname", "BMIz"]],
    on=["src_subject_id", "eventname"],
    how="left"
)

In [12]:
abcd_y_lt_clean

,src_subject_id,eventname,site_id_l,interview_age,family_id,BMIz
0,NDAR_INV003RTV85,baseline_year_1_arm_1,site06,131.0,8781.0,0.961662
1,NDAR_INV003RTV85,2_year_follow_up_y_arm_1,site06,157.0,8781.0,NaN
2,NDAR_INV005V6D2C,baseline_year_1_arm_1,site10,121.0,10210.0,1.462009
3,NDAR_INV005V6D2C,2_year_follow_up_y_arm_1,site10,148.0,10210.0,NaN
4,NDAR_INV007W6H7B,baseline_year_1_arm_1,site22,126.0,4722.0,0.550824
...,...,...,...,...,...,...
27590,NDAR_INVZZZNB0XC,baseline_year_1_arm_1,site03,108.0,6676.0,0.836305
27591,NDAR_INVZZZNB0XC,4_year_follow_up_y_arm_1,site03,157.0,6676.0,0.788166
27592,NDAR_INVZZZP87KR,baseline_year_1_arm_1,site19,126.0,7569.0,1.721165
27593,NDAR_INVZZZP87KR,2_year_follow_up_y_arm_1,site19,150.0,7569.0,1.882753


# Puberty

https://data-dict.abcdstudy.org/

Pubertal Development Scale (PDS)
Petersen, A. C., Crockett, L., et al. (1988) A self-report measure of pubertal status: Reliability, validity, and initial norms. J Youth Adolesc 17(2): 117-133.

In [13]:
# Load dataframes

# child report
ph_y_pds = pd.read_csv(datadir+'abcd-data-release-5.1/core/physical-health/ph_y_pds.csv')

# parent report
ph_p_pds = pd.read_csv(datadir+'abcd-data-release-5.1/core/physical-health/ph_p_pds.csv')

## Clean & Get PDS categories

### based on sex assigned at birth (what I am using for now)
- pds_y_ss_female_category_2
- pds_y_ss_male_cat_2

FYI the following variables are for PDS M/F based on self-identification question (i.e., gender) - I don't think it makes sense to use this unless there is HRT - and probably to few so if I find any, exclude
- pds_y_ss_female_category
- pds_y_ss_male_category

### categories
- 1 - prepuberty
- 2 - early puberty
- 3 - mid puberty
- 4 - late puberty
- 5 - post puberty

In [14]:
## Clean & get all the variables needed

# Create a new column with the reformatted IDs to match IDs in MPI folders
ph_y_pds['src_subject_id_fmt'] = ph_y_pds['src_subject_id'].apply(lambda x: 'sub-' + x.replace('_INV', 'INV'))
ph_p_pds['src_subject_id_fmt'] = ph_p_pds['src_subject_id'].apply(lambda x: 'sub-' + x.replace('_INV', 'INV'))

# take a subset of the dataframe - i.e. only the events we need: baseline, 2y and 4y follow up
ph_y_pds_full = ph_y_pds[ph_y_pds['eventname'].isin(['baseline_year_1_arm_1', '2_year_follow_up_y_arm_1', '4_year_follow_up_y_arm_1'])]
ph_p_pds_full = ph_p_pds[ph_p_pds['eventname'].isin(['baseline_year_1_arm_1', '2_year_follow_up_y_arm_1', '4_year_follow_up_y_arm_1'])]

# only select the variables we need (to compute PDS score (and to have PDS category) for male and female)
pds_y_selected = ph_y_pds_full[['src_subject_id_fmt', 'src_subject_id', 'eventname', 'pds_y_ss_female_category_2', 'pds_y_ss_male_cat_2', 'pds_ht2_y', 'pds_skin2_y', 'pds_bdyhair_y', 'pds_m4_y', 'pds_m5_y', 'pds_f4_2_y', 'pds_f5_y']]
pds_p_selected = ph_p_pds_full[['src_subject_id_fmt', 'src_subject_id', 'eventname', 'pds_p_ss_female_category_2', 'pds_p_ss_male_category_2', 'pds_1_p', 'pds_2_p', 'pds_3_p', 'pds_m4_p', 'pds_m5_p', 'pds_f4_p', 'pds_f5b_p']]

# merge with dataset containing age in months / family ID / site ID + rename columns
abcd_puberty = pd.merge(abcd_y_lt_clean, pds_y_selected, on=['src_subject_id', 'eventname'])
abcd_puberty = abcd_puberty.rename(columns={"interview_age": "age_months", "site_id_l": "site_id", "rel_family_id": "family_id"})

# merge successively also the parent report inside
abcd_puberty = pd.merge(abcd_puberty, pds_p_selected, on=['src_subject_id', 'eventname', 'src_subject_id_fmt'])

# make a column (separate for y and p) that will be pubertal CATEGORY (regardless of sex) - for analyses 
abcd_puberty['pds_y_cat'] = np.where(pd.isna(abcd_puberty.pds_y_ss_female_category_2), abcd_puberty.pds_y_ss_male_cat_2, abcd_puberty.pds_y_ss_female_category_2)
abcd_puberty['pds_p_cat'] = np.where(pd.isna(abcd_puberty.pds_p_ss_female_category_2), abcd_puberty.pds_p_ss_male_category_2, abcd_puberty.pds_p_ss_female_category_2)

# adding the biological sex + age at baseline from demographics dataframe 
abcd_puberty = pd.merge(abcd_demo_clean_baseline, abcd_puberty, on=['src_subject_id'])

# To move the fmt subject column to the left
abcd_puberty = abcd_puberty.set_index('src_subject_id_fmt')  # Set 'subject_id' as the index
abcd_puberty = abcd_puberty.reset_index() # Reset the index if you want 'subject_id' back as a column


## Calculate PDS numerical score (MEAN - when minimum 3 out 5 PDS questions are filled)

Youth
- M & F: 'pds_ht2_y', 'pds_skin2_y', 'pds_bdyhair_y´ -> ht (growth spurt), skin (skin changes, eg pimples), bdyhair (growth of body hair)
- M: 'pds_m4_y', 'pds_m5_y' -> m4 (voice), m5 (facial hair)
- F: 'pds_f4_2_y', 'pds_f5_y' -> f4_2 (breasts begun to grow), f5 (menstruation begin)
    - breasts growth (1-4: stages, 999: idk, 777: ? - pending data dictionary)
    - begun to menstruate (1: no, 4: yes, 999: idk, 777: ? - pending data dictionary), note that  'pds_f6_y' and 'pds_f6_y_dk' are "how old when began to menstruate"

Parent
- M & F: 'pds_1_p', 'pds_2_p', 'pds_3_p' -> 1 (growth spurt), 2 (growth of body hair), 3 (skin changes, eg pimples),
- M: 'pds_m4_p', 'pds_m5_p' -> m4 (voice), m5 (facial hair)
- F: 'pds_f4_p', 'pds_f5b_p' -> f4 (breasts begun to grow), f5b (menstruation begin)
    - breasts growth (1-4: stages, 999: idk, 777: ? - pending data dictionary)
    - begun to menstruate (1: no, 4: yes, 999: idk, 777: ? - pending data dictionary), note that  'pds_f6_p' and 'pds_f6_p_dk' are "how old when began to menstruate"

In [15]:
# Define variable names for males and females (y)
male_vars_y = ['pds_ht2_y', 'pds_skin2_y', 'pds_bdyhair_y', 'pds_m4_y', 'pds_m5_y']
female_vars_y = ['pds_ht2_y', 'pds_skin2_y', 'pds_bdyhair_y', 'pds_f4_2_y', 'pds_f5_y']
unique_vars_y = ['pds_ht2_y', 'pds_skin2_y', 'pds_bdyhair_y', 'pds_m4_y', 'pds_m5_y', 'pds_f4_2_y', 'pds_f5_y']  # "unique" is for data manipulation below where I cannot call on the same column twice

# Define variable names for males and females (p)
male_vars_p = ['pds_1_p', 'pds_2_p', 'pds_3_p', 'pds_m4_p', 'pds_m5_p']
female_vars_p = ['pds_1_p', 'pds_2_p', 'pds_3_p', 'pds_f4_p', 'pds_f5b_p']
unique_vars_p = ['pds_1_p', 'pds_2_p', 'pds_3_p', 'pds_m4_p', 'pds_m5_p', 'pds_f4_p', 'pds_f5b_p']  # "unique" is for data manipulation below where I cannot call on the same column twice

# Convert 999 and 777 to actual NaN values for computation
abcd_puberty.loc[:, unique_vars_y + unique_vars_p] = abcd_puberty.loc[:, unique_vars_y + unique_vars_p].replace([999, "999", 777, "777"], np.nan)

# Convert all variables to numeric (if they were strings)
abcd_puberty[unique_vars_y + unique_vars_p] = abcd_puberty[unique_vars_y + unique_vars_p].apply(pd.to_numeric, errors='coerce')

In [16]:
### 1. Calculate Male & Female specific PDS score (MEAN) - seprately for Youth and Parent reports

## Youth

# Males
n_valid_m_y = abcd_puberty[male_vars_y].notna().sum(axis=1)  # Count number of non-NaN values per row in male_vars_y
mean_m_y = abcd_puberty[male_vars_y].mean(axis=1, skipna=True)  # Compute the mean across male_vars_y (skipping NaNs)
abcd_puberty['pds_m_y_score'] = np.where(
    (abcd_puberty['sex'] == 'M') & (n_valid_m_y >= 3),  # Apply condition: sex == 'M' AND at least 3 non-NaN values
    mean_m_y,
    np.nan
)

# Females
n_valid_f_y = abcd_puberty[female_vars_y].notna().sum(axis=1)
mean_f_y = abcd_puberty[female_vars_y].mean(axis=1, skipna=True)
abcd_puberty['pds_f_y_score'] = np.where(
    (abcd_puberty['sex'] == 'F') & (n_valid_f_y >= 3),
    mean_f_y,
    np.nan
)


## Parent

# Males
n_valid_m_p = abcd_puberty[male_vars_p].notna().sum(axis=1)
mean_m_p = abcd_puberty[male_vars_p].mean(axis=1, skipna=True)
abcd_puberty['pds_m_p_score'] = np.where(
    (abcd_puberty['sex'] == 'M') & (n_valid_m_p >= 3),
    mean_m_p,
    np.nan
)

# Females
n_valid_f_p = abcd_puberty[female_vars_p].notna().sum(axis=1)
mean_f_p = abcd_puberty[female_vars_p].mean(axis=1, skipna=True)
abcd_puberty['pds_f_p_score'] = np.where(
    (abcd_puberty['sex'] == 'F') & (n_valid_f_p >= 3),
    mean_f_p,
    np.nan
)


### 2. Make a column (separately for Youth and Parent reports) that will be pubertal SCORE (regardless of sex) - for analyses 

# Youth
abcd_puberty['pds_y_score'] = np.where(pd.isna(abcd_puberty.pds_f_y_score), abcd_puberty.pds_m_y_score, abcd_puberty.pds_f_y_score)

# Parent
abcd_puberty['pds_p_score'] = np.where(pd.isna(abcd_puberty.pds_f_p_score), abcd_puberty.pds_m_p_score, abcd_puberty.pds_f_p_score)


In [17]:
abcd_puberty

,src_subject_id_fmt,src_subject_id,age_baseline_yrs,edu_parent_self,edu_parent_partner,sex,edu_parent,eventname,site_id,age_months,...,pds_f4_p,pds_f5b_p,pds_y_cat,pds_p_cat,pds_m_y_score,pds_f_y_score,pds_m_p_score,pds_f_p_score,pds_y_score,pds_p_score
0,sub-NDARINV003RTV85,NDAR_INV003RTV85,10.0,13.0,13.0,F,13.0,baseline_year_1_arm_1,site06,131.0,...,3.0,1.0,3.0,3.0,NaN,1.600000,NaN,2.4,1.600000,2.4
1,sub-NDARINV003RTV85,NDAR_INV003RTV85,10.0,13.0,13.0,F,13.0,2_year_follow_up_y_arm_1,site06,157.0,...,3.0,4.0,4.0,4.0,NaN,2.600000,NaN,2.8,2.600000,2.8
2,sub-NDARINV005V6D2C,NDAR_INV005V6D2C,10.0,6.0,NaN,F,6.0,baseline_year_1_arm_1,site10,121.0,...,3.0,1.0,NaN,3.0,NaN,NaN,NaN,2.0,NaN,2.0
3,sub-NDARINV005V6D2C,NDAR_INV005V6D2C,10.0,6.0,NaN,F,6.0,2_year_follow_up_y_arm_1,site10,148.0,...,3.0,4.0,5.0,4.0,NaN,3.800000,NaN,3.0,3.800000,3.0
4,sub-NDARINV007W6H7B,NDAR_INV007W6H7B,10.0,19.0,18.0,M,19.0,baseline_year_1_arm_1,site22,126.0,...,NaN,NaN,2.0,1.0,1.5,NaN,1.0,NaN,1.500000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27590,sub-NDARINVZZZNB0XC,NDAR_INVZZZNB0XC,9.0,19.0,13.0,F,19.0,baseline_year_1_arm_1,site03,108.0,...,1.0,1.0,2.0,1.0,NaN,1.250000,NaN,1.4,1.250000,1.4
27591,sub-NDARINVZZZNB0XC,NDAR_INVZZZNB0XC,9.0,19.0,13.0,F,19.0,4_year_follow_up_y_arm_1,site03,157.0,...,2.0,1.0,3.0,3.0,NaN,2.000000,NaN,1.8,2.000000,1.8
27592,sub-NDARINVZZZP87KR,NDAR_INVZZZP87KR,10.0,17.0,13.0,F,17.0,baseline_year_1_arm_1,site19,126.0,...,3.0,1.0,NaN,3.0,NaN,2.333333,NaN,2.6,2.333333,2.6
27593,sub-NDARINVZZZP87KR,NDAR_INVZZZP87KR,10.0,17.0,13.0,F,17.0,2_year_follow_up_y_arm_1,site19,150.0,...,3.0,4.0,4.0,4.0,NaN,2.800000,NaN,3.2,2.800000,3.2


In [18]:
# split into dataframes by wave of data: 3 timepoints -> baseline, 2y, and 4y
# subset by eventname and keep only desired columns
# (not sure I'll keep these dataframes separate but for now they are what I use for plotting (easier))

columns_to_keep = ['src_subject_id_fmt', 'age_baseline_yrs', 'sex', 'edu_parent', 'eventname', 'site_id', 'family_id', 'age_months','pds_y_cat', 'pds_p_cat', 'pds_y_score', 'pds_p_score', 'BMIz']

abcd_puberty_baseline = abcd_puberty.loc[
    abcd_puberty['eventname'] == 'baseline_year_1_arm_1',
    columns_to_keep
]

abcd_puberty_fu2y = abcd_puberty.loc[
    abcd_puberty['eventname'] == '2_year_follow_up_y_arm_1',
    columns_to_keep
]

abcd_puberty_fu4y = abcd_puberty.loc[
    abcd_puberty['eventname'] == '4_year_follow_up_y_arm_1',
    columns_to_keep
]


# export -> for preliminary analyses
abcd_puberty_baseline.to_csv(datadir_local+'abcd_puberty_baseline.csv', header = True, index = False)
abcd_puberty_fu2y.to_csv(datadir_local+'abcd_puberty_fu2y.csv', header = True, index = False)
abcd_puberty_fu4y.to_csv(datadir_local+'abcd_puberty_fu4y.csv', header = True, index = False)

abcd_puberty_baseline.to_csv(datadir+'abcd_puberty_baseline.csv', header = True, index = False)
abcd_puberty_fu2y.to_csv(datadir+'abcd_puberty_fu2y.csv', header = True, index = False)
abcd_puberty_fu4y.to_csv(datadir+'abcd_puberty_fu4y.csv', header = True, index = False)

In [19]:
abcd_puberty_baseline

,src_subject_id_fmt,age_baseline_yrs,sex,edu_parent,eventname,site_id,family_id,age_months,pds_y_cat,pds_p_cat,pds_y_score,pds_p_score,BMIz
0,sub-NDARINV003RTV85,10.0,F,13.0,baseline_year_1_arm_1,site06,8781.0,131.0,3.0,3.0,1.600000,2.4,0.961662
2,sub-NDARINV005V6D2C,10.0,F,6.0,baseline_year_1_arm_1,site10,10210.0,121.0,NaN,3.0,NaN,2.0,1.462009
4,sub-NDARINV007W6H7B,10.0,M,19.0,baseline_year_1_arm_1,site22,4722.0,126.0,2.0,1.0,1.500000,1.0,0.550824
6,sub-NDARINV00BD7VDC,9.0,M,20.0,baseline_year_1_arm_1,site07,3810.0,112.0,2.0,2.0,1.800000,1.6,0.007539
8,sub-NDARINV00CY2MDM,10.0,M,15.0,baseline_year_1_arm_1,site20,5355.0,130.0,2.0,2.0,2.000000,1.6,1.060263
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27581,sub-NDARINVZZNX6W2P,10.0,M,18.0,baseline_year_1_arm_1,site14,3797.0,131.0,1.0,1.0,1.200000,1.0,-0.395219
27584,sub-NDARINVZZPKBDAC,9.0,F,19.0,baseline_year_1_arm_1,site12,2445.0,113.0,NaN,3.0,2.250000,2.0,0.672931
27587,sub-NDARINVZZZ2ALR6,10.0,F,21.0,baseline_year_1_arm_1,site08,7032.0,121.0,2.0,1.0,1.600000,1.2,-1.708612
27590,sub-NDARINVZZZNB0XC,9.0,F,19.0,baseline_year_1_arm_1,site03,6676.0,108.0,2.0,1.0,1.250000,1.4,0.836305


In [20]:
abcd_puberty_fu2y

,src_subject_id_fmt,age_baseline_yrs,sex,edu_parent,eventname,site_id,family_id,age_months,pds_y_cat,pds_p_cat,pds_y_score,pds_p_score,BMIz
1,sub-NDARINV003RTV85,10.0,F,13.0,2_year_follow_up_y_arm_1,site06,8781.0,157.0,4.0,4.0,2.6,2.8,NaN
3,sub-NDARINV005V6D2C,10.0,F,6.0,2_year_follow_up_y_arm_1,site10,10210.0,148.0,5.0,4.0,3.8,3.0,NaN
7,sub-NDARINV00BD7VDC,9.0,M,20.0,2_year_follow_up_y_arm_1,site07,3810.0,141.0,1.0,1.0,1.0,1.0,NaN
9,sub-NDARINV00CY2MDM,10.0,M,15.0,2_year_follow_up_y_arm_1,site20,5355.0,152.0,2.0,3.0,1.6,2.2,0.735892
12,sub-NDARINV00HEV6HB,10.0,M,13.0,2_year_follow_up_y_arm_1,site12,2257.0,149.0,3.0,2.0,2.4,1.8,-0.617092
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27579,sub-NDARINVZZLZCKAY,9.0,F,15.0,2_year_follow_up_y_arm_1,site06,9347.0,131.0,5.0,4.0,3.8,3.4,2.255128
27582,sub-NDARINVZZNX6W2P,10.0,M,18.0,2_year_follow_up_y_arm_1,site14,3797.0,156.0,2.0,1.0,1.4,1.2,-0.304549
27585,sub-NDARINVZZPKBDAC,9.0,F,19.0,2_year_follow_up_y_arm_1,site12,2445.0,136.0,4.0,4.0,3.2,3.2,0.860740
27588,sub-NDARINVZZZ2ALR6,10.0,F,21.0,2_year_follow_up_y_arm_1,site08,7032.0,145.0,3.0,3.0,2.0,2.6,-1.390836


In [21]:
abcd_puberty_fu4y

,src_subject_id_fmt,age_baseline_yrs,sex,edu_parent,eventname,site_id,family_id,age_months,pds_y_cat,pds_p_cat,pds_y_score,pds_p_score,BMIz
5,sub-NDARINV007W6H7B,10.0,M,19.0,4_year_follow_up_y_arm_1,site21,4722.0,177.0,4.0,4.0,3.0,3.2,NaN
10,sub-NDARINV00CY2MDM,10.0,M,15.0,4_year_follow_up_y_arm_1,site20,5355.0,181.0,3.0,4.0,2.6,3.0,1.688158
13,sub-NDARINV00HEV6HB,10.0,M,13.0,4_year_follow_up_y_arm_1,site12,2257.0,173.0,3.0,3.0,2.6,2.4,0.128652
18,sub-NDARINV00LH735Y,9.0,M,13.0,4_year_follow_up_y_arm_1,site03,6069.0,156.0,3.0,4.0,2.8,3.2,NaN
30,sub-NDARINV00X2TBWJ,10.0,F,18.0,4_year_follow_up_y_arm_1,site14,3692.0,186.0,4.0,4.0,3.8,3.6,0.366317
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27583,sub-NDARINVZZNX6W2P,10.0,M,18.0,4_year_follow_up_y_arm_1,site14,3797.0,184.0,3.0,2.0,2.0,1.8,-0.089101
27586,sub-NDARINVZZPKBDAC,9.0,F,19.0,4_year_follow_up_y_arm_1,site12,2445.0,158.0,5.0,4.0,3.8,3.4,1.361156
27589,sub-NDARINVZZZ2ALR6,10.0,F,21.0,4_year_follow_up_y_arm_1,site08,7032.0,169.0,4.0,4.0,3.4,3.2,-1.696446
27591,sub-NDARINVZZZNB0XC,9.0,F,19.0,4_year_follow_up_y_arm_1,site03,6676.0,157.0,3.0,3.0,2.0,1.8,0.788166


#
# CBCL

In [86]:
# Load cbcl data
abcd_cbcl = pd.read_csv(datadir+'abcd-data-release-5.1/core/mental-health/mh_p_cbcl.csv')
abcd_cbcl

/tmp/ipykernel_136591/4211934653.py:2: DtypeWarning: Columns (124,128,132,136,140,144,148,152,156,160,164,168,172,176,180,184,188,192,196,200) have mixed types. Specify dtype option on import or set low_memory=False.
  abcd_cbcl = pd.read_csv(datadir+'abcd-data-release-5.1/core/mental-health/mh_p_cbcl.csv')


,src_subject_id,eventname,cbcl_select_language___1,cbcl_q01_p,cbcl_q02_p,cbcl_q03_p,cbcl_q04_p,cbcl_q05_p,cbcl_q06_p,cbcl_q07_p,...,cbcl_scr_07_sct_nm,cbcl_scr_07_ocd_r,cbcl_scr_07_ocd_t,cbcl_scr_07_ocd_m,cbcl_scr_07_ocd_nm,cbcl_scr_07_stress_r,cbcl_scr_07_stress_t,cbcl_scr_07_stress_m,cbcl_scr_07_stress_nm,cbcl_scr_07_stress_nm_2
0,NDAR_INV003RTV85,baseline_year_1_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
1,NDAR_INV003RTV85,1_year_follow_up_y_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
2,NDAR_INV003RTV85,2_year_follow_up_y_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
3,NDAR_INV003RTV85,3_year_follow_up_y_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
4,NDAR_INV005V6D2C,baseline_year_1_arm_1,1,0,0,0,0,0,0,0,...,0.0,1.0,51.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48732,NDAR_INVZZZP87KR,baseline_year_1_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
48733,NDAR_INVZZZP87KR,1_year_follow_up_y_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
48734,NDAR_INVZZZP87KR,2_year_follow_up_y_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN
48735,NDAR_INVZZZP87KR,3_year_follow_up_y_arm_1,0,0,0,0,0,0,0,0,...,0.0,0.0,50.0,NaN,0.0,0.0,50.0,NaN,0.0,NaN


In [87]:
## Clean & get all the variables needed

# Create a new column with the reformatted IDs to match IDs in MPI folders
abcd_cbcl['src_subject_id_fmt'] = abcd_cbcl['src_subject_id'].apply(lambda x: 'sub-' + x.replace('_INV', 'INV'))

# take a subset of the dataframe - i.e. only the events we need: baseline, 2y and 4y follow up
abcd_cbcl = abcd_cbcl[abcd_cbcl['eventname'].isin(['baseline_year_1_arm_1', '2_year_follow_up_y_arm_1', '4_year_follow_up_y_arm_1'])]

# only select the variables we need (sub IDs, eventname, and CBCL total, internalizing, externalizing)
abcd_cbcl = abcd_cbcl[['src_subject_id_fmt', 'eventname','cbcl_scr_syn_totprob_r', 'cbcl_scr_syn_internal_r', 'cbcl_scr_syn_external_r']]

# rename some variables
abcd_cbcl = abcd_cbcl.rename(columns={"cbcl_scr_syn_totprob_r": "cbcl_tot", "cbcl_scr_syn_internal_r": "cbcl_int", "cbcl_scr_syn_external_r": "cbcl_ext"})

# To move the fmt subject column to the left
abcd_cbcl = abcd_cbcl.set_index('src_subject_id_fmt')  # Set 'subject_id' as the index
abcd_cbcl = abcd_cbcl.reset_index() # Reset the index if you want 'subject_id' back as a column

In [88]:
# split into dataframes by wave of data (subset by eventname): 3 timepoints -> baseline, 2y, and 4y 
abcd_cbcl_baseline = abcd_cbcl.loc[abcd_cbcl['eventname'] == 'baseline_year_1_arm_1']
abcd_cbcl_fu2y = abcd_cbcl.loc[abcd_cbcl['eventname'] == '2_year_follow_up_y_arm_1']
abcd_cbcl_fu4y = abcd_cbcl.loc[abcd_cbcl['eventname'] == '4_year_follow_up_y_arm_1']


# export -> for preliminary analyses
abcd_cbcl_baseline.to_csv(datadir_local+'abcd_cbcl_baseline.csv', header = True, index = False)
abcd_cbcl_fu2y.to_csv(datadir_local+'abcd_cbcl_fu2y.csv', header = True, index = False)
abcd_cbcl_fu4y.to_csv(datadir_local+'abcd_cbcl_fu4y.csv', header = True, index = False)

In [89]:
abcd_cbcl

,src_subject_id_fmt,eventname,cbcl_tot,cbcl_int,cbcl_ext
0,sub-NDARINV003RTV85,baseline_year_1_arm_1,4.0,1.0,1.0
1,sub-NDARINV003RTV85,2_year_follow_up_y_arm_1,3.0,1.0,1.0
2,sub-NDARINV005V6D2C,baseline_year_1_arm_1,2.0,1.0,0.0
3,sub-NDARINV005V6D2C,2_year_follow_up_y_arm_1,0.0,0.0,0.0
4,sub-NDARINV007W6H7B,baseline_year_1_arm_1,23.0,8.0,2.0
...,...,...,...,...,...
27433,sub-NDARINVZZZNB0XC,baseline_year_1_arm_1,12.0,2.0,4.0
27434,sub-NDARINVZZZNB0XC,4_year_follow_up_y_arm_1,8.0,3.0,0.0
27435,sub-NDARINVZZZP87KR,baseline_year_1_arm_1,6.0,3.0,0.0
27436,sub-NDARINVZZZP87KR,2_year_follow_up_y_arm_1,1.0,0.0,0.0


In [90]:
abcd_cbcl[abcd_cbcl["cbcl_tot"].isna()]

,src_subject_id_fmt,eventname,cbcl_tot,cbcl_int,cbcl_ext
313,sub-NDARINV0AU5R8NA,2_year_follow_up_y_arm_1,NaN,NaN,NaN
2875,sub-NDARINV37MKKJBX,4_year_follow_up_y_arm_1,NaN,NaN,NaN
8451,sub-NDARINV9PVR76W7,baseline_year_1_arm_1,NaN,NaN,NaN
14606,sub-NDARINVGYHXT5DV,2_year_follow_up_y_arm_1,NaN,NaN,NaN
15972,sub-NDARINVJHJDGEFN,baseline_year_1_arm_1,NaN,NaN,NaN


#
# Adversity: Family Environment Scale (FES)

In [18]:
# Load cbcl data
abcd_fes = pd.read_csv(datadir+'abcd-data-release-5.1/core/culture-environment/ce_y_fes.csv')
abcd_fes

,src_subject_id,eventname,fes_youth_q10,fes_youth_q1,fes_youth_q11,fes_youth_q2,fes_youth_q12,fes_youth_q3,fes_youth_q13,fes_youth_q4,...,fes_youth_q7,fes_youth_q17,fes_youth_q8,fes_youth_q18,fes_youth_q9,fes_y_ss_fc,fes_y_ss_fc_nm,fes_y_ss_fc_nt,fes_y_ss_fc_na,fes_y_ss_fc_pr
0,NDAR_INV003RTV85,baseline_year_1_arm_1,NaN,1.0,NaN,0.0,NaN,0.0,NaN,1.0,...,0.0,NaN,0.0,NaN,0.0,3.0,0,9,9,3.0
1,NDAR_INV003RTV85,1_year_follow_up_y_arm_1,NaN,0.0,NaN,1.0,NaN,0.0,NaN,1.0,...,1.0,NaN,0.0,NaN,1.0,4.0,0,9,9,4.0
2,NDAR_INV003RTV85,2_year_follow_up_y_arm_1,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,...,1.0,NaN,0.0,NaN,0.0,2.0,0,9,9,2.0
3,NDAR_INV003RTV85,3_year_follow_up_y_arm_1,NaN,0.0,NaN,1.0,NaN,0.0,NaN,1.0,...,0.0,NaN,0.0,NaN,1.0,3.0,0,9,9,3.0
4,NDAR_INV005V6D2C,baseline_year_1_arm_1,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,0.0,0,9,9,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49146,NDAR_INVZZZP87KR,baseline_year_1_arm_1,NaN,1.0,NaN,1.0,NaN,1.0,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,5.0,0,9,9,5.0
49147,NDAR_INVZZZP87KR,1_year_follow_up_y_arm_1,NaN,1.0,NaN,1.0,NaN,0.0,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,2.0,0,9,9,2.0
49148,NDAR_INVZZZP87KR,2_year_follow_up_y_arm_1,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,...,0.0,NaN,1.0,NaN,0.0,2.0,0,9,9,2.0
49149,NDAR_INVZZZP87KR,3_year_follow_up_y_arm_1,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,...,0.0,NaN,1.0,NaN,1.0,8.0,0,9,9,8.0


In [19]:
## Clean & get all the variables needed

# Create a new column with the reformatted IDs to match IDs in MPI folders
abcd_fes['src_subject_id_fmt'] = abcd_fes['src_subject_id'].apply(lambda x: 'sub-' + x.replace('_INV', 'INV'))

# take a subset of the dataframe - i.e. only the events we need: baseline, 2y and 4y follow up
abcd_fes = abcd_fes[abcd_fes['eventname'].isin(['baseline_year_1_arm_1', '2_year_follow_up_y_arm_1', '4_year_follow_up_y_arm_1'])]

# only select the variables we need (sub IDs, eventname, and family conflic mean score)
abcd_fes = abcd_fes[['src_subject_id_fmt', 'eventname','fes_y_ss_fc', 'fes_y_ss_fc_nm']]

# rename some variables
abcd_fes = abcd_fes.rename(columns={"fes_y_ss_fc": "fam_conflict_y", "fes_y_ss_fc_nm": "num_missing"})

# take a subset that only include rows where there are no nm
abcd_fes = abcd_fes[abcd_fes["num_missing"] == 0]

# drop the num_missing column
abcd_fes = abcd_fes.drop(columns=["num_missing"])

# To move the fmt subject column to the left
abcd_fes = abcd_fes.set_index('src_subject_id_fmt')  # Set 'subject_id' as the index
abcd_fes = abcd_fes.reset_index() # Reset the index if you want 'subject_id' back as a column

In [20]:
abcd_fes

,src_subject_id_fmt,eventname,fam_conflict_y
0,sub-NDARINV003RTV85,baseline_year_1_arm_1,3.0
1,sub-NDARINV003RTV85,2_year_follow_up_y_arm_1,2.0
2,sub-NDARINV005V6D2C,baseline_year_1_arm_1,0.0
3,sub-NDARINV005V6D2C,2_year_follow_up_y_arm_1,4.0
4,sub-NDARINV007W6H7B,baseline_year_1_arm_1,0.0
...,...,...,...
27514,sub-NDARINVZZZNB0XC,baseline_year_1_arm_1,3.0
27515,sub-NDARINVZZZNB0XC,4_year_follow_up_y_arm_1,0.0
27516,sub-NDARINVZZZP87KR,baseline_year_1_arm_1,5.0
27517,sub-NDARINVZZZP87KR,2_year_follow_up_y_arm_1,2.0


In [21]:
# split into dataframes by wave of data (subset by eventname): 3 timepoints -> baseline, 2y, and 4y 
abcd_fes_baseline = abcd_fes.loc[abcd_fes['eventname'] == 'baseline_year_1_arm_1']
abcd_fes_fu2y = abcd_fes.loc[abcd_fes['eventname'] == '2_year_follow_up_y_arm_1']
abcd_fes_fu4y = abcd_fes.loc[abcd_fes['eventname'] == '4_year_follow_up_y_arm_1']


# export -> for preliminary analyses
abcd_fes_baseline.to_csv(datadir_local+'abcd_fes_baseline.csv', header = True, index = False)
abcd_fes_fu2y.to_csv(datadir_local+'abcd_fes_fu2y.csv', header = True, index = False)
abcd_fes_fu4y.to_csv(datadir_local+'abcd_fes_fu4y.csv', header = True, index = False)

#
#
#
#
#
#
